# Baseline Model



In [27]:
# imports
import io
import numpy as np
import librosa
import soundfile as sf
from datasets import Audio, load_dataset
from datasets import load_dataset_builder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [2]:
# login Hugging Face Hub
import os
from dotenv import load_dotenv

load_dotenv()  
HF_TOKEN = os.getenv("HF_TOKEN")

from huggingface_hub import login

login(token=HF_TOKEN) 

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [13]:
# inspect dataset
ds_builder = load_dataset_builder("SleepyJesse/ai_music_tiny")
ds_builder.info.description
ds_builder.info.features

{'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None),
 'ai_generated': Value('bool'),
 'source': Value('string')}

In [28]:
# load dataset
# ds = load_dataset("SleepyJesse/ai_music_tiny", split="train", cache_dir="./data")
ds = load_dataset("SleepyJesse/ai_music_tiny", split="train")
# Avoid torchcodec decoding in environments where FFmpeg/TorchCodec isn't configured.
ds = ds.cast_column("audio", Audio(decode=False))
ds

Dataset({
    features: ['audio', 'ai_generated', 'source'],
    num_rows: 1000
})

Sampling and data processing

In [29]:
def _resolve_label_key(example, preferred=None):
    if preferred is not None and preferred in example:
        return preferred

    for candidate in ("ai_generated", "label", "target", "class"):
        if candidate in example:
            return candidate

    raise KeyError(
        f"Could not find a label column. Available keys: {list(example.keys())}"
    )


def sample_per_class(dataset, label_key=None, samples_per_class=500):
    class_counts = {}
    selected_indices = []
    expected_labels = None

    if hasattr(dataset, "features") and label_key is None:
        if "ai_generated" in dataset.features:
            label_key = "ai_generated"
        elif "label" in dataset.features:
            label_key = "label"

    if hasattr(dataset, "features") and label_key in getattr(dataset, "features", {}):
        label_feature = dataset.features[label_key]
        if hasattr(label_feature, "names") and label_feature.names is not None:
            expected_labels = set(range(len(label_feature.names)))
        elif getattr(label_feature, "dtype", None) == "bool":
            expected_labels = {False, True}

    # For Hugging Face Dataset objects, iterate over label column only to avoid audio decoding.
    if hasattr(dataset, "num_rows") and hasattr(dataset, "select"):
        if label_key is None:
            first_example = dataset[0]
            label_key = _resolve_label_key(first_example)

        labels = dataset[label_key]
        for idx, label in enumerate(labels):
            if label not in class_counts:
                class_counts[label] = 0

            if class_counts[label] < samples_per_class:
                selected_indices.append(idx)
                class_counts[label] += 1

            if expected_labels is not None and all(
                class_counts.get(lbl, 0) >= samples_per_class for lbl in expected_labels
            ):
                break

        return dataset.select(selected_indices)

    for example in dataset:
        label_key = _resolve_label_key(example, preferred=label_key)
        label = example[label_key]

        if label not in class_counts:
            class_counts[label] = 0

        if class_counts[label] < samples_per_class:
            selected_indices.append(example)
            class_counts[label] += 1

        if expected_labels is not None and all(
            class_counts.get(lbl, 0) >= samples_per_class for lbl in expected_labels
        ):
            break

    return selected_indices

In [30]:
def segment_audio(audio_array, sr, clip_duration=5):
    clip_length = sr * clip_duration
    segments = []

    for start in range(0, len(audio_array), clip_length):
        end = start + clip_length
        segment = audio_array[start:end]

        if len(segment) < clip_length:
            break

        segments.append(segment)

    return segments

Extract mel frequency cepstral coefficients

In [31]:
def extract_mfcc_features(audio, sr, n_mfcc=20):
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=n_mfcc
    )

    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std = mfcc.std(axis=1)

    return np.concatenate([mfcc_mean, mfcc_std])

Build dataset

In [38]:
def _load_audio_from_entry(audio_entry):
    if isinstance(audio_entry, dict):
        if audio_entry.get("array") is not None and audio_entry.get("sampling_rate") is not None:
            return np.asarray(audio_entry["array"]), int(audio_entry["sampling_rate"])

        if audio_entry.get("bytes") is not None:
            audio, sr = sf.read(io.BytesIO(audio_entry["bytes"]))
            if getattr(audio, "ndim", 1) > 1:
                audio = np.mean(audio, axis=1)
            return np.asarray(audio), int(sr)

        if audio_entry.get("path"):
            audio, sr = librosa.load(audio_entry["path"], sr=None, mono=True)
            return audio, int(sr)

    if isinstance(audio_entry, str):
        audio, sr = librosa.load(audio_entry, sr=None, mono=True)
        return audio, int(sr)

    raise ValueError("Unsupported audio format in dataset row.")


def build_feature_dataset(dataset_subset, label_key=None):

    X = []
    y = []

    for example in dataset_subset:
        resolved_label_key = _resolve_label_key(example, preferred=label_key)
        label = example[resolved_label_key]

        if "audio" not in example:
            raise KeyError("Expected an 'audio' column in the dataset.")

        audio, sr = _load_audio_from_entry(example["audio"])
        segments = segment_audio(audio, sr)

        for seg in segments:
            features = extract_mfcc_features(seg, sr)

            X.append(features)
            y.append(label)

    return np.array(X), np.array(y)

Train model

In [33]:
subset = sample_per_class(ds, label_key="ai_generated", samples_per_class=500)

In [39]:
X, y = build_feature_dataset(subset)

print("Feature shape:", X.shape)

Feature shape: (32527, 40)


In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

model = LogisticRegression(max_iter=2000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [41]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8339993851829081
              precision    recall  f1-score   support

       False       0.86      0.94      0.90      4995
        True       0.71      0.48      0.57      1511

    accuracy                           0.83      6506
   macro avg       0.78      0.71      0.73      6506
weighted avg       0.82      0.83      0.82      6506

